# 对PepSet结构预测后的通过的结果进行三次LigandMPNN设计，并对设计结果进行筛选与预测。计算相较于设计前的预测结构的scRMSD

In [ ]:
import os
import pandas as pd
import json

with open('/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_filter/datasets/PepSet-passed-dimer_processed/PDB.list', 'r') as f:
    pdbs = [line.strip() for line in f if line.strip()]

for pdb in pdbs:
    base_dir = f"/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_filter/Protenix/2000-0.1T-dimer/{pdb}/ligandmpnn_seed44"
    metrics_path = f"{base_dir}/metrics_summary.csv"
    output_path = f"{base_dir}/metrics_summary-pep_plddt.csv"
    
    new_metrics = []
    new_columns = ['complex', 'seed', 'id', 'pep_plddt', 'gpde', 'ptm', 'iptm', 'ranking_score', 'scRMSD']
    with open(metrics_path, 'r') as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        if i >= 1:
            parts = line.strip().split(',')
            complex = parts[0]
            seed = parts[1]
            id = parts[2]
            gpde = float(parts[4])
            ptm = float(parts[5])
            iptm = float(parts[6])
            ranking_score = float(parts[7])
            scRMSD = float(parts[8])

            json_path = f"{base_dir}/{complex}/{seed}/predictions/{complex}_summary_confidence_sample_{id}.json"
            json_file = open(json_path, 'r')
            data = json.load(json_file)
            pep_plddt = round(data['chain_plddt'][1], 4)
            # print(f"{complex}, {seed}, {id}, {pep_plddt}")
            new_metrics.append([complex, seed, id, pep_plddt, gpde, ptm, iptm, ranking_score, scRMSD])
    result = pd.DataFrame(new_metrics, columns=new_columns)
    result.to_csv(output_path, index=False)
    

    # result_filtered1 = result[result['scRMSD'] <= 1.0].reset_index(drop=True)
    # result_filtered1.to_csv(f"{base_dir}/metrics_filtered_scRMSD_le1-pep_plddt.csv", index=False)
    # result_filtered15 = result[result['scRMSD'] <= 1.5].reset_index(drop=True)
    # result_filtered15.to_csv(f"{base_dir}/metrics_filtered_scRMSD_le1.5-pep_plddt.csv", index=False)
    # result_filtered2 = result[result['scRMSD'] <= 2.0].reset_index(drop=True)
    # result_filtered2.to_csv(f"{base_dir}/metrics_filtered_scRMSD_le2-pep_plddt.csv", index=False)
    # result_filtered25 = result[result['scRMSD'] <= 2.5].reset_index(drop=True)
    # result_filtered25.to_csv(f"{base_dir}/metrics_filtered_scRMSD_le2.5-pep_plddt.csv", index=False)

In [42]:
import os
import pandas as pd

cutoffs = [1, 1.5, 2, 2.5]
seeds = ["ligandmpnn_seed42", "ligandmpnn_seed43", "ligandmpnn_seed44"]

pep_plddt_cutoff = 0.7
iptm_cutoff = 0.7

str_cutoff = "0" + str(int(100 * pep_plddt_cutoff))

with open('/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_filter/datasets/PepSet-passed-dimer_processed/PDB_pep_plddt.list', 'r') as f:
    pdbs = [line.strip() for line in f if line.strip()]

pdb_w = max(4, max(len(p) for p in pdbs))
col_w = 8
block_w = col_w * len(cutoffs) + (len(cutoffs) - 1)

def format_block(values):
    return " ".join([str(v).rjust(col_w) for v in values])

header1 = f"{'PDB':<{pdb_w}}  " + " | ".join([seed.center(block_w) for seed in seeds])
header2 = f"{'':<{pdb_w}}  " + " | ".join([format_block([f"{c:g}" for c in cutoffs]) for _ in seeds])

print(header1)
print(header2)

csv_rows = []
csv_header = ["pdb"] + [f"{seed}_cutoff_{c:g}" for seed in seeds for c in cutoffs]
csv_rows.append(csv_header)

for pdb in pdbs:
    seed_blocks = []
    row = [pdb]
    for seed in seeds:
        df_path = f'./Protenix/2000-0.1T-dimer/{pdb}/{seed}/metrics_summary-pep_plddt.csv'
        if not os.path.exists(df_path):
            seed_blocks.append(format_block(["NA"] * len(cutoffs)))
            row.extend(["NA"] * len(cutoffs))
            continue
        df = pd.read_csv(df_path)
        oricount = df['complex'].nunique()
        rates = []
        for cutoff in cutoffs:
            count = df[(df['pep_plddt'] > pep_plddt_cutoff) & (df['iptm'] > iptm_cutoff) & (df['scRMSD'] < cutoff)]['complex'].nunique()
            rate = count / oricount * 100
            rate_str = f"{rate:.2f}"
            rates.append(rate_str)
            row.append(rate_str)
        seed_blocks.append(format_block(rates))
    csv_rows.append(row)
    print(f"{pdb:<{pdb_w}}  " + " | ".join(seed_blocks))

df_out = pd.DataFrame(csv_rows[1:], columns=csv_rows[0])
mean_series = df_out.drop(columns=['pdb']).replace('NA', pd.NA).astype(float).mean()
mean_row = ['MEAN'] + [f"{mean_series[col]:.2f}" if pd.notna(mean_series[col]) else "NA" for col in df_out.columns[1:]]
csv_rows.append(mean_row)
print(f"{'MEAN':<{pdb_w}}  " + " | ".join([format_block(mean_row[1 + i * len(cutoffs):1 + (i + 1) * len(cutoffs)]) for i in range(len(seeds))]))
csv_output_path = f'./Protenix/2000-0.1T-dimer/success_rates_by_plddt_{str_cutoff}-new.csv'
pd.DataFrame(csv_rows[1:], columns=csv_rows[0]).to_csv(csv_output_path, index=False)
print(f"CSV saved to: {csv_output_path}")

PDB                            ligandmpnn_seed42          |          ligandmpnn_seed43          |          ligandmpnn_seed44         
                             1      1.5        2      2.5 |        1      1.5        2      2.5 |        1      1.5        2      2.5
1nrl_sample_1_seed43    100.00   100.00   100.00   100.00 |   100.00   100.00   100.00   100.00 |   100.00   100.00   100.00   100.00
MEAN                    100.00   100.00   100.00   100.00 |   100.00   100.00   100.00   100.00 |   100.00   100.00   100.00   100.00
CSV saved to: ./Protenix/2000-0.1T-dimer/success_rates_by_plddt_070-new.csv


In [ ]:
import os
import pandas as pd

cutoff = 1.5
seed = "ligandmpnn_seed42"


with open('/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_filter/datasets/PepSet-passed-dimer_processed/PDB.list', 'r') as f:
    pdbs = [line.strip() for line in f if line.strip()]
# print(pdbs)
success_sum = 0
overall_sum = 0
for pdb in pdbs:
    fil_df = pd.read_csv(f'./Protenix/2000-0.1T-dimer/{pdb}/{seed}/metrics_filtered_scRMSD_le{cutoff}-pep_plddt.csv')
    df = pd.read_csv(f'./Protenix/2000-0.1T-dimer/{pdb}/{seed}/metrics_summary.csv')
    # 筛选fil_df中plddt大于0.85，iptm大于0.7的行中complex不重复的个数，将其保存在一个字典中，key为pdb，value为个数
    count = fil_df[(fil_df['pep_plddt'] > 0.85) & (fil_df['iptm'] > 0.7)]['complex'].nunique()
    oricount = df['complex'].nunique()
    success_sum += count
    overall_sum += oricount
    print(f"{pdb}: {count:>2}, decoys before filtering: {oricount:>2}, success rate(%): {count/oricount*100:.2f}")
overall_success_rate = success_sum / overall_sum if overall_sum > 0 else 0
print(f"Overall success rate: {overall_success_rate:.2%}")

# 检查每一个结构预测的15个结构的序列是否json文件的序列保持一致

In [12]:
# 检查/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_filter/Protenix/2000-0.1T-dimer下每一个pdb的ligandmpnn_seed目录中每一个job name的所有seed的预测结果的B链是否和pred.json的多肽链一致
import os
import json
from Bio.PDB import PDBParser, FastMMCIFParser
from Bio.SeqUtils import seq1

with open('/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_filter/datasets/PepSet-passed-dimer_processed/PDB.list', 'r') as f:
    pdbs = [line.strip() for line in f if line.strip()]

mmcifparser = FastMMCIFParser(QUIET=True)
for pdb in pdbs:
    pep_base_dir = f"/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_filter/Protenix/2000-0.1T-dimer/{pdb}"
    seed_dirs = [d for d in os.listdir(pep_base_dir) if d.startswith('ligandmpnn_seed44')]
    for seed in seed_dirs:
        pred_json_path = f'{pep_base_dir}/{seed}/pred.json'
        if not os.path.exists(pred_json_path):
            print(f"pred.json not found for {pdb} {seed}")
            continue
        with open(pred_json_path, 'r') as f:
            jobs = json.load(f)
        updated_jobs = []
        for job in jobs:
           job_name = job['name']
           for protenix_seed in ['seed_42', 'seed_43', 'seed_44']:
                cif_path = f'{pep_base_dir}/{seed}/{job_name}/{protenix_seed}/predictions/{job_name}_sample_0.cif'
                if not os.path.exists(cif_path):
                   continue
                structure_pred = mmcifparser.get_structure('pred', cif_path)
                chain_B_pred = structure_pred[0]['B']
                peptide_sequence = ''.join([seq1(residue.get_resname()) for residue in chain_B_pred.get_residues() if residue.get_id()[0] == ' '])
                if peptide_sequence != job['sequences'][1]['proteinChain']['sequence']:
                    print(f"Sequence mismatch for {pdb} {seed} {job_name} {protenix_seed}: pred.json sequence: {job['sequences'][1]['proteinChain']['sequence']}, CIF sequence: {peptide_sequence}")
                else:
                    print(f"Sequence match for {pdb} {seed} {job_name} {protenix_seed}:pred.json sequence: {job['sequences'][1]['proteinChain']['sequence']}, CIF sequence: {peptide_sequence} ")

Sequence match for 1czy_sample_1 ligandmpnn_seed44 1czy_sample_1_1 seed_42:pred.json sequence: ATRASGA, CIF sequence: ATRASGA 
Sequence match for 1czy_sample_1 ligandmpnn_seed44 1czy_sample_1_1 seed_43:pred.json sequence: ATRASGA, CIF sequence: ATRASGA 
Sequence match for 1czy_sample_1 ligandmpnn_seed44 1czy_sample_1_1 seed_44:pred.json sequence: ATRASGA, CIF sequence: ATRASGA 
Sequence match for 1czy_sample_1 ligandmpnn_seed44 1czy_sample_1_2 seed_42:pred.json sequence: ATRASGT, CIF sequence: ATRASGT 
Sequence match for 1czy_sample_1 ligandmpnn_seed44 1czy_sample_1_2 seed_43:pred.json sequence: ATRASGT, CIF sequence: ATRASGT 
Sequence match for 1czy_sample_1 ligandmpnn_seed44 1czy_sample_1_2 seed_44:pred.json sequence: ATRASGT, CIF sequence: ATRASGT 
Sequence match for 1czy_sample_1 ligandmpnn_seed44 1czy_sample_1_3 seed_42:pred.json sequence: ATRASST, CIF sequence: ATRASST 
Sequence match for 1czy_sample_1 ligandmpnn_seed44 1czy_sample_1_3 seed_43:pred.json sequence: ATRASST, CIF seq